1. Import libraries

In [31]:
# Import library
import pickle
import json
import dill
import pandas as pd
import numpy as np

2. Load all the necessary pipeline, functions, and columns

In [32]:
# Load model
with open('pipelines_inference.pkl', 'rb') as file_1:
  pipelines_inference = pickle.load(file_1)

with open('num_col.txt', 'r') as file_2:
  num_col = json.load(file_2)

with open('cat_col.txt', 'r') as file_3:
  cat_col = json.load(file_3)

with open('preprocessing_function.pkl', 'rb') as file_4:
    preprocess_inference_data = dill.load(file_4)

3. Create the inference data into a list of dictionary (inference data 1 and inference data 2)

In [33]:
data_inf = {
    'ID': [43, 8999, 89141, 20120091, 20124537],
    'Building ID': [138, 115, 151, 128, 137],
    'Site ID': [1, 1, 1, 1, 1],
    'Building Use': ['Public services', 'Education', 'Office', 'Lodging/residential', 'Entertainment/public assembly'],
    'Building Area (sqft)': [118231, 129716, 30495, 102774, 64024],
    'Year Built': [float('nan'), 1968, 1997, 1956, 1967],
    'Floor Count': [6, 6, 8, 7, 6],
    'Meter Type': [3, 0, 0, 0, 0],
    'Time Stamp': ['2016-01-01 00:00:00', '2016-01-01 09:00:00', '2016-01-04 09:00:00', '2016-12-31 19:00:00', '2016-12-31 23:00:00'],
    'Air Temperature (°C)': [3.8, 2.2, 7.7, 8.1, 7.5],
    'Cloud Coverage (oktas)': [float('nan'), float('nan'), float('nan'), float('nan'), float('nan')],
    'Dew Temperature (°C)': [2.4, 1.2, 6.5, 6.5, 6.5],
    'Precipitation Depth (mm/hr)': [float('nan'), float('nan'), float('nan'), float('nan'), float('nan')],
    'Sea Level Pressure (mbar/hPa)': [1020.9, 1022.0, 982.8, 1027.5, 1024.0],
    'Wind Direction (degree)': [240.0, 100.0, 160.0, 220.0, 210.0],
    'Wind Speed (m/s)': [3.1, 2.6, 4.1, 3.6, 5.7]
}

data_inf = pd.DataFrame(data_inf)
data_inf

,ID,Building ID,Site ID,Building Use,Building Area (sqft),Year Built,Floor Count,Meter Type,Time Stamp,Air Temperature (°C),Cloud Coverage (oktas),Dew Temperature (°C),Precipitation Depth (mm/hr),Sea Level Pressure (mbar/hPa),Wind Direction (degree),Wind Speed (m/s)
0,43,138,1,Public services,118231,NaN,6,3,2016-01-01 00:00:00,3.8,NaN,2.4,NaN,1020.9,240.0,3.1
1,8999,115,1,Education,129716,1968.0,6,0,2016-01-01 09:00:00,2.2,NaN,1.2,NaN,1022.0,100.0,2.6
2,89141,151,1,Office,30495,1997.0,8,0,2016-01-04 09:00:00,7.7,NaN,6.5,NaN,982.8,160.0,4.1
3,20120091,128,1,Lodging/residential,102774,1956.0,7,0,2016-12-31 19:00:00,8.1,NaN,6.5,NaN,1027.5,220.0,3.6
4,20124537,137,1,Entertainment/public assembly,64024,1967.0,6,0,2016-12-31 23:00:00,7.5,NaN,6.5,NaN,1024.0,210.0,5.7


4. Apply new categorized features into the data inference.

In [34]:
# Apply categorized features
data_inf = preprocess_inference_data(data_inf)

In [35]:
#check the new data inference
data_inf

,ID,Building ID,Site ID,Building Use,Building Area (sqft),Year Built,Floor Count,Meter Type,Time Stamp,Air Temperature (°C),Cloud Coverage (oktas),Dew Temperature (°C),Precipitation Depth (mm/hr),Sea Level Pressure (mbar/hPa),Wind Direction (degree),Wind Speed (m/s),Time,Time Category,Era Category,Meter Type Category
0,43,138,1,Public services,118231,NaN,6,3,2016-01-01 00:00:00,3.8,NaN,2.4,NaN,1020.9,240.0,3.1,00:00:00,Night,None,Hot Water
1,8999,115,1,Education,129716,1968.0,6,0,2016-01-01 09:00:00,2.2,NaN,1.2,NaN,1022.0,100.0,2.6,09:00:00,Morning/Day,Pre-Energy Code Era (before 1981),Electricity
2,89141,151,1,Office,30495,1997.0,8,0,2016-01-04 09:00:00,7.7,NaN,6.5,NaN,982.8,160.0,4.1,09:00:00,Morning/Day,Post-Energy Code Era (after 1980),Electricity
3,20120091,128,1,Lodging/residential,102774,1956.0,7,0,2016-12-31 19:00:00,8.1,NaN,6.5,NaN,1027.5,220.0,3.6,19:00:00,Afternoon/Evening,Pre-Energy Code Era (before 1981),Electricity
4,20124537,137,1,Entertainment/public assembly,64024,1967.0,6,0,2016-12-31 23:00:00,7.5,NaN,6.5,NaN,1024.0,210.0,5.7,23:00:00,Night,Pre-Energy Code Era (before 1981),Electricity


5. Check the pipeline for data inference

In [36]:
pipelines_inference

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', MinMaxScaler(),
                                                  ['Building Area (sqft)',
                                                   'Floor Count',
                                                   'Air Temperature (°C)',
                                                   'Dew Temperature (°C)',
                                                   'Wind Speed (m/s)']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Building Use',
                                                   'Time Category',
                                                   'Era Category',
                                                   'Meter Type Category'])])),
                ('regressor',
                 HistGradientBoostingRegressor(l2_regularization=0.1,
                                               max_depth=7, max_iter=200,
                                               random_state=22))])

6. Predicts the data inference on their meter readings.

In [37]:
print("="*100)
print("METER READING (KWH) PREDICTION")
print("="*100)

# predict
y_pred_inf = pipelines_inference.predict(data_inf)

# results
for i, pred in enumerate(y_pred_inf):
    print(f"Data {i+1} | {data_inf['Year Built'].iloc[i]} | {data_inf['Building Use'].iloc[i]} | {data_inf['Time Stamp'].iloc[i].strftime('%H:%M:%S')} | meter readings: {pred:,.0f} kwh")
    print()

METER READING (KWH) PREDICTION
Data 1 | nan | Public services | 00:00:00 | meter readings: 21 kwh

Data 2 | 1968.0 | Education | 09:00:00 | meter readings: 469 kwh

Data 3 | 1997.0 | Office | 09:00:00 | meter readings: 55 kwh

Data 4 | 1956.0 | Lodging/residential | 19:00:00 | meter readings: 115 kwh

Data 5 | 1967.0 | Entertainment/public assembly | 23:00:00 | meter readings: 46 kwh

